In [1]:
import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"]="6"
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import math
import time
import scanpy as sc
from tqdm.notebook import tqdm
import gc
import random
import copy
import pickle
import warnings
warnings.filterwarnings("ignore")

from gANCHOR.module2_response import PatientResponseModel, response_train
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, multilabel_confusion_matrix

from matplotlib import rcParams
rcParams['figure.figsize']=(10, 10)

In [2]:
PATH = '../results'

model_name = 'gANCHOR_2026-08-04_02:53:18'
fn = f'{PATH}/{model_name}'

In [3]:
m = sc.read_h5ad(f"{fn}/gANCHOR_moduleI_outputs.h5ad")
m

AnnData object with n_obs × n_vars = 1693727 × 11140
    obs: 'cell_type', 'assay', 'patient', 'sample_id', 'data_type', 'batch_correct', 'Response', 'sample_id_old', 'percent_ribo', 'n_counts', 'barcode'
    var: 'gene_median'
    uns: 'data_yuniques', 'log1p', 'patients_split', 'type_yuniques'
    obsm: 'cell_embed', 'gene_pred', 'type_prob'

In [4]:
train_patients, valid_patients, test_patients = m.uns['patients_split']['train'], m.uns['patients_split']['val'], m.uns['patients_split']['test'] 
len(train_patients), len(valid_patients), len(test_patients)

(101, 26, 34)

In [5]:
max_cell_num = 2000

random.seed(0)
cell_pick = []
cell_unpick = []
for i in list(train_patients) + list(valid_patients) + list(test_patients):
    cell_patients = m[m.obs['patient'] == i].obs_names.tolist()
    if len(cell_patients) <= max_cell_num:
        cell_pick += cell_patients
    else:
        selected = random.sample(cell_patients, max_cell_num)
        unselected = list(set(cell_patients) - set(selected))
        cell_pick += selected
        cell_unpick += unselected
split_cell_map = {idx: 'pick' for idx in cell_pick}
split_cell_map.update({idx: 'unpick' for idx in cell_unpick})
m.obs['response_pick'] = m.obs.index.map(split_cell_map)
print(len(cell_pick), len(cell_unpick))
m

301296 473737


AnnData object with n_obs × n_vars = 1693727 × 11140
    obs: 'cell_type', 'assay', 'patient', 'sample_id', 'data_type', 'batch_correct', 'Response', 'sample_id_old', 'percent_ribo', 'n_counts', 'barcode', 'response_pick'
    var: 'gene_median'
    uns: 'data_yuniques', 'log1p', 'patients_split', 'type_yuniques'
    obsm: 'cell_embed', 'gene_pred', 'type_prob'

In [7]:
device = torch.device('cuda')

epochs = 1000
scheduler_patience = 50
NR_R_weight = [3.0, 1.0]
factor = 0.1
eps = 1e-08
early_stopping = 120

seeds = [1, 2, 3, 4]
lrs = [0.00001, 0.00005, 0.0001, 0.0005, 0.001, 0.005, 0.01]
weight_decays = [0.01, 0.001, 0.0001]

saved_results = []

for seed in tqdm(seeds, desc="Runs", leave=False):
    for weight_decay in tqdm(weight_decays, desc="Weight decay", leave=False):
        for lr in tqdm(lrs, desc="Learning rate", leave=False):

            np.random.seed(seed)
            
            model = PatientResponseModel(m.obsm['cell_embed'].shape[1], max_cell=max_cell_num, device=device, fc_dropout=0.2, seed=seed)
            model = model.to(device)

            optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay, eps=eps)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=factor, patience=scheduler_patience)
            
            (tr_pa_pred, val_pa_pred, test_pa_pred, tr_pa_GT, val_pa_GT, test_pa_GT, tr_pa_cell_wt, val_pa_cell_wt, test_pa_cell_wt, epoch) = \
            response_train(model, optimizer, m, train_patients, valid_patients, test_patients, max_cell=max_cell_num, scheduler=scheduler, device=device, epochs=epochs, patience=early_stopping, response_weights=NR_R_weight)

            _, tr_pred = torch.max(tr_pa_pred, 1)
            _, val_pred = torch.max(val_pa_pred, 1)
            _, test_pred = torch.max(test_pa_pred, 1)

            f1_tr = f1_score(tr_pa_GT, tr_pred, average='binary')
            f1_val = f1_score(val_pa_GT, val_pred, average='binary')
            f1_test = f1_score(test_pa_GT, test_pred, average='binary')

            saved_results.append([seed, lr, weight_decay, tr_pa_pred.tolist(), val_pa_pred.tolist(), test_pa_pred.tolist(), tr_pa_GT.tolist(), val_pa_GT.tolist(), test_pa_GT.tolist(), tr_pa_cell_wt.tolist(),  
                                  val_pa_cell_wt.tolist(), test_pa_cell_wt.tolist(), f1_macro_tr, f1_macro_val, f1_macro_test])

Runs:   0%|          | 0/4 [00:00<?, ?it/s]

Weight decay:   0%|          | 0/3 [00:00<?, ?it/s]

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

...

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

Weight decay:   0%|          | 0/3 [00:00<?, ?it/s]

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

In [9]:
from datetime import date
from time import gmtime, strftime

saved_time = strftime("%Y-%m-%d_%H:%M:%S", gmtime())
fn1 = f"{fn}/response_{saved_time}"

if os.path.exists(fn1) is not True:
    os.mkdir(fn1)
    
saved_results_table = pd.DataFrame([row for row in saved_results], columns=['seed', 'lr', 'we_de', 'tr_pred', 'val_pred', 'test_pred', 'tr_GT', 'val_GT', 
                                                                            'test_GT', 'tr_cell_wt', 'val_cell_wt', 'test_cell_wt', 'f1_tr', 'f1_val', 'f1_test'])

file = open(f'{fn1}/gANCHOR_moduleII_response_outputs.pkl','wb')
pickle.dump(saved_results_table, file, protocol=4)
file.close()

In [10]:
m.obs[['barcode', 'response_pick']].to_csv(f'{fn1}/response_cell_pick.csv', index=True, header=True)